# Conformal prediction on Matt's subset (YOLO crops + masks)

**Pre-trained models only — no training.** We load the networks already trained in `benchmark.ipynb` (on the NIH malaria dataset) and run conformal prediction on Matt's subset for **evaluation only**. The subset is never used for training.

Run the **binary conformal classifier** (Pipeline 1) on the subset of 75 images:
- **Uninfected:** red blood cells (and leukocytes if present).
- **Infected:** gametocyte, ring, schizont, trophozoite.

**Benchmark:** For each saved model in `models/` (SimpleCNN, MobileNetV2, ResNet50V2), we (1) load its pre-trained weights, (2) calibrate **Crepes** on the NIH calibration set (to set the conformal threshold; this does not update the network), (3) run conformal prediction on the subset, (4) report coverage and avg set size per model.

Steps:
1. Load subset images and metadata (from a configurable path).
2. Resize to 128×128 (same as benchmark).
3. Load **pre-trained** models from `models/`; for each, calibrate Crepes on NIH calibration data and evaluate on the subset.
4. Report **benchmark summary** and **by-stage** error analysis for the paper.

In [1]:
# Path to subset: folder containing metadata.csv and subfolders infected/ and uninfected/
# Default: CONFORMAL_SUBSET_PATH env var, or /Users/beylouni/Downloads/images
# To use project data: SUBSET_ROOT = PROJECT_ROOT / "data" / "external" / "conformal_subset"
import os
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()  # run notebook from notebooks/
SUBSET_ROOT = Path(os.environ.get("CONFORMAL_SUBSET_PATH", "/Users/beylouni/Downloads/images"))
METADATA_PATH = SUBSET_ROOT / "metadata.csv"
MODELS_DIR = PROJECT_ROOT / "models"
# Run conformal prediction for each of these if the saved model exists (e.g. from benchmark.ipynb)
MODEL_NAMES = ["SimpleCNN", "MobileNetV2", "ResNet50V2"]
IMG_SIZE = 128
CONFIDENCE = 0.95  # 1 - alpha for conformal prediction sets

print(f"Subset root: {SUBSET_ROOT}")
print(f"Metadata:   {METADATA_PATH} (exists: {METADATA_PATH.exists()})")
for name in MODEL_NAMES:
    p = MODELS_DIR / f"{name}_best.keras"
    print(f"  {name}: {p} (exists: {p.exists()})")

Subset root: /Users/beylouni/Downloads/images
Metadata:   /Users/beylouni/Downloads/images/metadata.csv (exists: True)
  SimpleCNN: /Users/beylouni/Documents/conformalized-xplainable-malaria/models/SimpleCNN_best.keras (exists: True)
  MobileNetV2: /Users/beylouni/Documents/conformalized-xplainable-malaria/models/MobileNetV2_best.keras (exists: True)
  ResNet50V2: /Users/beylouni/Documents/conformalized-xplainable-malaria/models/ResNet50V2_best.keras (exists: True)


In [2]:
import numpy as np
import pandas as pd
import tensorflow as tf
import keras
from keras.layers import Input, Dense, GlobalAveragePooling2D, Dropout, Conv2D, MaxPooling2D, Flatten
from keras.models import Model
from keras.applications import MobileNetV2, ResNet50V2
from keras.optimizers import Adam
from crepes import WrapClassifier
from PIL import Image

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)
print(f"TensorFlow {tf.__version__}, Keras {keras.__version__}")

TensorFlow 2.10.0, Keras 2.10.0


In [3]:
# KerasWrapper for Crepes (same as benchmark)
class KerasWrapper:
    def __init__(self, model):
        self.model = model
        self.classes_ = np.array([0, 1])

    def fit(self, X, y, **kwargs):
        return self.model.fit(X, y, **kwargs)

    def predict(self, X):
        probs = self.model.predict(X, verbose=0)
        return (probs > 0.5).astype(int).flatten()

    def predict_proba(self, X):
        probs = self.model.predict(X, verbose=0)
        p1 = np.array(probs).flatten()
        return np.column_stack([1 - p1, p1])

In [4]:
def build_simple_cnn(input_shape=(IMG_SIZE, IMG_SIZE, 3), lr=1e-3):
    inputs = Input(shape=input_shape)
    x = keras.layers.Rescaling(1.0 / 255)(inputs)
    x = Conv2D(32, (3, 3), activation="relu")(x)
    x = MaxPooling2D((2, 2))(x)
    x = Conv2D(64, (3, 3), activation="relu")(x)
    x = MaxPooling2D((2, 2))(x)
    x = Flatten()(x)
    x = Dense(64, activation="relu")(x)
    outputs = Dense(1, activation="sigmoid")(x)
    model = Model(inputs, outputs, name="SimpleCNN")
    model.compile(optimizer=Adam(learning_rate=lr), loss="binary_crossentropy", metrics=["accuracy"])
    return model

def build_mobilenetv2(input_shape=(IMG_SIZE, IMG_SIZE, 3), lr=1e-4):
    inputs = Input(shape=input_shape)
    x = keras.layers.Rescaling(1.0 / 127.5, offset=-1, name="rescaling")(inputs)
    base_model = MobileNetV2(include_top=False, weights="imagenet", input_tensor=x)
    base_model.trainable = True
    for layer in base_model.layers[:-20]:
        layer.trainable = False
    x = base_model.output
    x = GlobalAveragePooling2D(name="gap")(x)
    x = Dropout(0.2, name="dropout")(x)
    outputs = Dense(1, activation="sigmoid", name="dense")(x)
    model = Model(inputs=inputs, outputs=outputs, name="MobileNetV2")
    model.compile(optimizer=Adam(learning_rate=lr), loss="binary_crossentropy", metrics=["accuracy"])
    return model

def build_resnet50v2(input_shape=(IMG_SIZE, IMG_SIZE, 3), lr=1e-4):
    inputs = Input(shape=input_shape)
    x = keras.layers.Rescaling(1.0 / 127.5, offset=-1)(inputs)
    base_model = ResNet50V2(input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights="imagenet")
    base_model.trainable = True
    for layer in base_model.layers[:-20]:
        layer.trainable = False
    x = base_model(x, training=False)
    x = GlobalAveragePooling2D(name="gap")(x)
    x = Dropout(0.2, name="dropout")(x)
    outputs = Dense(1, activation="sigmoid", name="dense")(x)
    model = Model(inputs, outputs, name="ResNet50V2")
    model.compile(optimizer=Adam(learning_rate=lr), loss="binary_crossentropy", metrics=["accuracy"])
    return model

MODEL_BUILDERS = {"SimpleCNN": build_simple_cnn, "MobileNetV2": build_mobilenetv2, "ResNet50V2": build_resnet50v2}

## 1. Load subset images and metadata

In [5]:
def load_subset(subset_root, metadata_path, img_size=128):
    """Load images and labels from subset folder. Expects metadata.csv and infected/ / uninfected/ subfolders."""
    meta = pd.read_csv(metadata_path).dropna(how="all")
    meta = meta[meta["filename"].astype(str).str.strip() != ""]
    label_map = {"uninfected": 0, "infected": 1}
    images, labels, filenames, stages = [], [], [], []
    for _, row in meta.iterrows():
        fname = row["filename"].strip()
        label_str = row["label"].strip().lower()
        stage = row.get("stage", "")
        if isinstance(stage, float):
            stage = ""
        y = label_map.get(label_str)
        if y is None:
            continue
        # Look for file in subset_root, then in infected/ and uninfected/
        candidates = [
            subset_root / fname,
            subset_root / "infected" / fname,
            subset_root / "uninfected" / fname,
        ]
        path = None
        for c in candidates:
            if c.exists():
                path = c
                break
        if path is None:
            print(f"Warning: not found {fname}")
            continue
        img = Image.open(path).convert("RGB")
        img = np.array(img)
        if img.ndim == 2:
            img = np.stack([img] * 3, axis=-1)
        img = tf.image.resize(img, (img_size, img_size)).numpy()
        img = img.astype(np.float32)
        images.append(img)
        labels.append(y)
        filenames.append(fname)
        stages.append(stage)
    X = np.stack(images, axis=0)
    y = np.array(labels, dtype=np.int32)
    return X, y, filenames, stages, meta

In [6]:
X_subset, y_subset, filenames_subset, stages_subset, meta_subset = load_subset(
    SUBSET_ROOT, METADATA_PATH, IMG_SIZE
)
print(f"Loaded {len(X_subset)} images. Shape: {X_subset.shape}, dtype: {X_subset.dtype}")
print(f"Labels: {np.bincount(y_subset)} (0=uninfected, 1=infected)")
pd.Series(stages_subset).value_counts()

Metal device set to: Apple M3

systemMemory: 16.00 GB
maxCacheSize: 5.33 GB

Loaded 75 images. Shape: (75, 128, 128, 3), dtype: float32
Labels: [37 38] (0=uninfected, 1=infected)


2026-02-14 20:57:00.169332: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:306] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-02-14 20:57:00.169435: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:272] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


red blood cell    37
ring              12
trophozoite       11
gametocyte        10
schizont           5
Name: count, dtype: int64

## 2. Load NIH calibration set (same split as benchmark)

Crepes needs a calibration set to compute prediction sets. We use the same 10% calibration split from the NIH malaria dataset as in the benchmark.

In [7]:
print("1. Iniciando teste de importação...")

import numpy as np
print("2. NumPy importado com sucesso!")

print("3. Tentando importar TensorFlow (isso pode travar)...")
import tensorflow as tf
print("4. TensorFlow importado com sucesso!")

print("5. Tentando importar TFDS...")
import tensorflow_datasets as tfds
print("6. TFDS importado com sucesso!")

1. Iniciando teste de importação...
2. NumPy importado com sucesso!
3. Tentando importar TensorFlow (isso pode travar)...
4. TensorFlow importado com sucesso!
5. Tentando importar TFDS...
6. TFDS importado com sucesso!


In [ ]:
import tensorflow_datasets as tfds

def get_calibration_data(img_size=128, batch_size=64, seed=42):
    def preprocess(image, label):
        image = tf.image.resize(image, (img_size, img_size))
        image = tf.cast(image, tf.float32)
        return image, label

    ds_calib_raw = tfds.load(
        "malaria",
        split="train[80%:90%]",
        as_supervised=True,
        shuffle_files=True,
    )
    ds_calib = ds_calib_raw.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    batched = ds_calib.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    images, labels = [], []
    for img, lbl in tfds.as_numpy(batched):
        images.append(img)
        labels.append(lbl)
    X = np.concatenate(images, axis=0)
    y = np.concatenate(labels, axis=0)
    return X, y

RuntimeError: Visible devices cannot be modified after being initialized

In [ ]:
X_calib, y_calib = get_calibration_data(IMG_SIZE, batch_size=64, seed=RANDOM_SEED)
print(f"Calibration set: {X_calib.shape[0]} images")

## 3. Benchmark: load each trained model, calibrate Crepes, run conformal prediction on subset

In [ ]:
class_names = ["uninfected", "infected"]
benchmark_rows = []
results_per_model = []

# Pre-trained models only: load from disk (trained in benchmark.ipynb on NIH). No training on subset.
for model_name in MODEL_NAMES:
    path = MODELS_DIR / f"{model_name}_best.keras"
    if not path.exists():
        print(f"Skip {model_name}: {path} not found")
        continue
    print(f"--- {model_name} ---")
    keras.backend.clear_session()
    try:
        model = keras.models.load_model(path)  # pre-trained weights
    except Exception as e:
        print(f"  load_model failed; building and loading weights: {e}")
        model = MODEL_BUILDERS[model_name]()
        model.load_weights(path)
    wrapper = KerasWrapper(model)
    cp = WrapClassifier(learner=wrapper)
    cp.calibrate(X=X_calib, y=y_calib)  # NIH calibration set only; subset not used for calibration
    y_sets = cp.predict_set(X=X_subset, confidence=CONFIDENCE)  # evaluate on subset (inference only)
    correct = [y_subset[i] in y_sets[i] for i in range(len(y_subset))]
    coverage = np.mean(correct)
    avg_set_size = np.mean([len(s) for s in y_sets])
    benchmark_rows.append({"model": model_name, "coverage": coverage, "avg_set_size": avg_set_size})
    pred_str = ["{" + ", ".join(class_names[i] for i in sorted(s)) + "}" for s in y_sets]
    for i in range(len(filenames_subset)):
        results_per_model.append({
            "filename": filenames_subset[i], "label": [class_names[y] for y in y_subset][i],
            "stage": stages_subset[i], "model": model_name,
            "prediction_set": pred_str[i], "in_set": correct[i],
        })
    print(f"  Coverage: {coverage:.4f}, avg set size: {avg_set_size:.4f}")

benchmark_df = pd.DataFrame(benchmark_rows)
if benchmark_df.empty:
    raise FileNotFoundError("No model files found. Run benchmark.ipynb to train and save at least one model.")
results = pd.DataFrame(results_per_model)

## 4. Benchmark summary (coverage and avg set size per model)

In [ ]:
benchmark_df.round(4)

## 5. Detailed results (per image, per model) and by-stage error analysis

In [ ]:
results

In [ ]:
# By-stage and by-model: coverage and error count
by_stage = results.groupby(["stage", "model"]).agg(
    n=("filename", "count"),
    coverage=("in_set", "mean"),
    errors=("in_set", lambda s: (~s).sum()),
).round(4)
by_stage

In [ ]:
# Save results for the paper / further analysis
out_dir = PROJECT_ROOT / "output" / "conformal_subset"
out_dir.mkdir(parents=True, exist_ok=True)
results.to_csv(out_dir / "conformal_subset_results.csv", index=False)
benchmark_df.to_csv(out_dir / "conformal_subset_benchmark.csv", index=False)
by_stage.to_csv(out_dir / "conformal_subset_by_stage.csv")
print(f"Saved to {out_dir}")
# By label and model (infected vs uninfected)
results.groupby(["label", "model"]).agg(n=("filename", "count"), coverage=("in_set", "mean")).round(4)

## Optional: Explainability (Grad-CAM)

To generate Grad-CAM for specific subset images, use the same model and `get_grad_cam_final` / `plot_grad_cam` from `benchmark.ipynb` (target layer for MobileNetV2: `("mobilenetv2_1.00_128", "out_relu")`). You can run that notebook or copy the Grad-CAM cell here and feed `X_subset[i:i+1]` for a chosen index `i`.